In [ ]:
# Iterative macrophage refinement round (v4).
import os, glob
import scanpy as sc
import anndata as ad
import pandas as pd

input_dir = "/home/yanjun3/atlas/heart_sn"
import scvi
import seaborn as sns
import torch
from rich import print
from scib_metrics.benchmark import Benchmarker
sc.set_figure_params(figsize=(4, 4),  facecolor="white",
    frameon=True,   dpi = 300, format = 'png')
sc.settings.figdir = "/home/yanjun3/atlas/heart_sn/figures_v4"
sns.set_theme()
torch.set_float32_matmul_precision("high")

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [ ]:
# Load data
adata = sc.read('/home/yanjun2/atlas_project/heart_sn_h5ad/sn_final.h5ad')

In [ ]:
adata.obs_names_make_unique()

In [ ]:
adata_mac = adata[~adata.obs["leiden_res0_5"].isin(["5","8"])].copy()

In [ ]:
adata_mac

In [ ]:
adata_mac

In [ ]:
import scvi
import seaborn as sns
import torch
from rich import print
from scib_metrics.benchmark import Benchmarker

In [ ]:
batch_key = "sample"

In [ ]:
sc.pl.dotplot(
    adata, 
    var_names=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2',
              'C1QA', 'C1QC', 'CD68', 'CD163', 'HLA-DRA', 
              'SPP1', 'APOE',
              'FCGR2A', 'FCGR2B',
              'CLSPN', 'STIL',
              
              'CD3D', 'CD3E',
              'FCER1A', 'CLEC10A', 'CLEC9A','ITGAX', 'XCR1', 'CCR2', 'HDAC9',
               'S100A8', 'CD14',
               "TNNT2","MYL2","TTN","ACTC1","TNNI3","MYH6","MYH7","RYR2","PLN","FN1"
              ],
    groupby='leiden_res0_5',
    dot_max = 0.8,
    standard_scale='var', 
    color_map='RdBu_r' ,
    
   save = '05_dotplot.png'
)

In [ ]:
# scVI integration
scvi.model.SCVI.setup_anndata(adata_mac, layer="counts", batch_key=batch_key)
adata_mac

In [ ]:
model_scvi = scvi.model.SCVI(adata_mac)
model_scvi

In [ ]:
model_scvi.view_anndata_setup()

In [ ]:
model_scvi = scvi.model.SCVI(adata_mac)
model_scvi

In [ ]:
import numpy as np
max_epochs_scvi = np.min([round((20000 / adata.n_obs) * 400), 400])
max_epochs_scvi

In [ ]:
model_scvi.train()

In [ ]:
adata_mac.obsm["X_scVI"] = model_scvi.get_latent_representation()

In [ ]:
#adata = sc.read('myeloid_scvi_v1.h5ad')

In [ ]:
# Neighbours, UMAP and clustering
sc.pp.neighbors(adata_mac, use_rep="X_scVI")
sc.tl.umap(adata_mac)

In [ ]:
adata_mac

In [ ]:
sc.tl.leiden(adata_mac, key_added="leiden_res0_3", resolution=0.3)
sc.pl.umap(
    adata_mac,
    color=["leiden_res0_3"]
    ,
    frameon=False,
    ncols=1, size = 30, legend_loc = 'on data',
    #save = '0_5_v3'
)

In [ ]:
sc.tl.leiden(adata_mac, key_added="leiden_res0_4", resolution=0.4)
sc.pl.umap(
    adata_mac,
    color=["leiden_res0_4"]
    ,
    frameon=False,
    ncols=1, size = 30, legend_loc = 'on data',
    #save = '0_5_v3'
)

In [ ]:
sc.pl.umap(
    adata_mac,
    color=["leiden_res0_3", "leiden_res0_4", "LYVE1", "NRP1", "C1QA", "C1QC","CD68", "TNNT2",
          "MRC1", "F13A1", "APOE", "STIL", "G2M_score", "S_score"]
    ,
    frameon=False,
    ncols=2, size = 30, legend_loc = 'on data',
#    save = '0_5_v3'
)

In [ ]:
sc.pl.dotplot(
    adata_mac, 
    var_names=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2', 'NRP1',
              'C1QA', 'C1QC', 'CD68', 'CD163', 'HLA-DRA', 
              'SPP1', 'APOE',
              'FCGR2A', 'FCGR2B',
              'CLSPN', 'STIL',
              'TNNT2',
              'CD3D', 'CD3E',
              'FCER1A', 'CLEC10A', 'CLEC9A','ITGAX', 'XCR1', 'CCR2', 'HDAC9', 'CD1C', 
               'MS4A6A', 'MS4A6E', 'S100A8', 'CD14',
               "TNNT2","MYL2","TTN","ACTC1","TNNI3","MYH6","MYH7","RYR2","PLN","FN1"
              ],
    groupby='leiden_res0_3',
    dot_max = 0.8,
    standard_scale='var', 
    color_map='RdBu_r' ,
    
   save = '03_dotplot.png'
)

In [ ]:
sc.tl.dendrogram(adata_mac, groupby="leiden_res0_3")

In [ ]:
# Differential expression
sc.tl.rank_genes_groups(adata_mac, groupby="leiden_res0_3", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata_mac, n_genes=10, standard_scale='var', 
                                save = 'res_0_3_topmarkers_v3'
                               )

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_mac, n_genes=20, standard_scale='var', 
                                save = 'res_0_3_topmarkers20_v3'
                               )

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv(f"{input_dir}/v4_res03.csv", index=False)

In [ ]:
adata_mac.write(
 '/home/yanjun2/atlas_project/heart_sn_h5ad/sn_final.h5ad'
)

In [ ]:
sc.pl.violin(
    adata_mac,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)


In [ ]:
marker_dict = {
    "Adipocytic": [
        "ACACB", "GPAM", "MGST1", "PDE3B", "PPARG",
        "ACSL1", "GHR", "SOX5", "LINC00598", "PLIN1"
    ],
    "Cardiac": [
        "RYR2", "TTN", "FGF12", "MYBPC3", "SORBS2",
        "MLIP", "CTNNA3", "SLC8A1", "PALLD", "RBM20"
    ],
    "Endocardial": [
        "ECE1", "HSPG2", "PLPP3", "NFIA", "LEPR",
        "ST6GALNAC3", "ODF2L", "ITGA10", "PBX1", "SELP"
    ],
    "Endothelial": [
        "PLA2G5", "HSPG2", "ID3", "TIE1", "ST6GALNAC3",
        "NEXN", "IFI44L", "ADGRL4", "ARHGAP29", "PALMD"
    ],
    "Fibroblast": [
        "KAZN", "PLA2G5", "SPATA6", "ECHDC2", "NFIA",
        "NEGR1", "ST6GALNAC3", "TGFBR3", "DDR2", "PBX1"
    ],
    "Lymphoid": [
        "PIK3CD", "PLA2G5", "HSPG2", "CLIC4", "RUNX3",
        "EPB41", "LCK", "SMAP2", "NFIA", "JAK1"
    ],
    "Mesothelial": [
        "ERRFI1", "ENO1", "NPPA", "TNFRSF8", "PDPN",
        "PLA2G2A", "PLA2G5", "CLIC4", "MACF1", "MFSD2A"
    ],
    "Neuronal": [
        "PLA2G5", "CLIC4", "GRIK3", "MACF1", "TRABD2B",
        "SPATA6", "DOCK7", "ROR1", "DNAJC6", "NEGR1"
    ],
    "Perivascular": [
        "HES4", "VPS13D", "PLA2G5", "HSPG2", "HEYL",
        "ST6GALNAC3", "PALMD", "NGF", "RGS5", "AL499616.1"
    ]
}

In [ ]:
sc.pl.dotplot(
    adata,
    var_names = marker_dict,
    groupby   = "leiden_res0_3",   # change to your obs column
    dendrogram = False,
   # save = 'disco_broad.png'
)

In [ ]:
marker_dict = {
    "CD14_monocytes": [
        "ZEB2", "RBM47", "SIPA1L1", "TBXAS1", "RBPJ",
        "VCAN", "PLXDC2", "MS4A6A", "SMAP2", "CD163",
        "DPYD", "FMN1", "TRPS1", "PTPRC", "AOAH"
    ],

    "CD16_monocytes": [
        "LYN", "PTPRC", "TCF7L2", "TBXAS1", "SAT1",
        "PRKCB", "RIPOR2", "AOAH", "ZEB2", "DOCK2",
        "MTSS1", "LYST", "SMAP2", "PTPRJ", "DOCK8"
    ],

    "Cycling_LYVE1_macrophages": [
        "F13A1", "RBPJ", "FMN1", "FRMD4B", "MRC1",
        "SLC9A9", "COLEC12", "RBM47", "CD163", "STAB1",
        "SYK", "MS4A6A", "ZEB2", "LRMDA", "ATP8B4"
    ],

    "LYVE1_macrophages": [
        "F13A1", "MRC1", "RBPJ", "FMN1", "CD163",
        "FRMD4B", "RBM47", "MAN1A1", "SLC9A9", "MS4A6A",
        "SLCO2B1", "COLEC12", "MERTK", "TBXAS1", "STAB1"
    ],

    "SPP1_macrophages": [
        "FMN1", "RBM47", "CPM", "CD163", "IQGAP2",
        "SLC1A3", "TBXAS1", "MSR1", "ATG7", "PLXDC2",
        "ZEB2", "MERTK", "MS4A6A", "RBPJ", "FRMD4B"
    ]
}

In [ ]:
sc.pl.dotplot(
    adata_mac,
    var_names = marker_dict,
    groupby   = "leiden_res0_3",   # change to your obs column
    dendrogram = False,
   # save = 'disco_fine.png'
)

In [ ]:
sc.pl.dotplot(
    adata, 
    var_names=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2',
              'C1QA', 'C1QC', 'CD68', 'CD163', 'HLA-DRA', 
              'SPP1', 'APOE',
              'FCGR2A', 'FCGR2B',
              'CLSPN', 'STIL',
              
              'CD3D', 'CD3E'],
    groupby='leiden_res0_5',
    dot_max = 0.8,
    standard_scale='var', 
    color_map='RdBu_r' ,
    
   save = '05_dotplot.png'
)

In [ ]:
s_genes = [
    "MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1", "UNG",
    "GINS2", "MCM6", "CDCA7", "DTL", "PRIM1", "UHRF1", "HELLS",
    "RFC2", "RPA2", "NASP", "RAD51AP1", "GMNN", "WDR76", "SLBP",
    "CCNE2", "UBL5", "POLD3", "MSH2", "ATAD2", "RAD51", "RRM2",
    "CDC45", "CDC6", "EXO1", "TIPIN", "DSCC1", "BLM", "CASP8AP2",
    "USP1", "CLSPN", "POLA1", "CHAF1B", "BRIP1", "E2F8"
]


In [ ]:
g2m_genes = [
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A",
    "NDC80", "CKS2", "NUF2", "CKS1B", "MKI67", "TMPO", "CENPF",
    "TACC3", "FAM64A", "SMC4", "CCNB2", "CKAP2L", "CKAP2",
    "AURKB", "BUB1", "KIF11", "ANP32E", "TUBB4B", "GTSE1",
    "KIF20B", "HJURP", "CDCA3", "HN1", "CDC20", "TTK", "CDC25C",
    "KIF2C", "RANGAP1", "NCAPD2", "DLGAP5", "CDCA2", "CDCA8",
    "ECT2", "KIF23", "HMMR", "AURKA", "PSRC1", "ANLN", "LBR",
    "CKAP5", "CENPE", "CTCF", "NEK2", "G2E3", "GAS2L3", "CBX5",
    "CENPA"
]


In [ ]:
# Gene scoring
sc.tl.score_genes_cell_cycle(
    adata,
    s_genes=s_genes,
    g2m_genes=g2m_genes
)


In [ ]:
sc.pl.violin(adata, ["S_score", "G2M_score"], groupby="leiden_res0_5")

In [ ]:
sc.pl.umap(adata, color = ["S_score", "G2M_score"] ,
          size = 50)

In [ ]:
# Save result
adata.write('macrophagesv2.h5ad'
)

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby='leiden_res0_5',
    groups=["0"],       # test group
    reference="3",      # comparison group
    method="wilcoxon",
    use_raw=False
)

In [ ]:
res = adata.uns["rank_genes_groups"]

names = res["names"]["0"]
scores = res["scores"]["0"]
pvals = res["pvals"]["0"]
pvals_adj = res["pvals_adj"]["0"]
logfc = res["logfoldchanges"]["0"]

import pandas as pd

df0_vs_1 = pd.DataFrame({
    "gene": names,
    "score": scores,
    "pval": pvals,
    "pval_adj": pvals_adj,
    "logFC": logfc
})
df0_vs_1.to_csv(f"{input_dir}/DEG_cluster0_vs_cluster3.csv", index=False)

In [ ]:
sc.pl.umap(
    adata,
    color = [
    'LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2',
    'C1QA', 'C1QC', 'CD68', 'CD163', 'HLA-DRA',
    'SPP1', 'APOE',
    'FCGR2A', 'FCGR2B',
    'CLSPN', 'STIL',
    'CD3D', 'CD3E', 'STAB1', 'NEAT1', 'AP2A2', 'leiden_res0_5'
],
    ncols = 2,
    size = 30,
    save = 'manygenes_umap'
)

In [ ]:
gene_list = [
    "MAN1A1", "CSGALNACT1", "PDE4D", "CPM", "PDGFC", "MAMDC2", "SELENOP",
    "ARHGAP24", "MS4A6A", "SESN1", "SCN9A", "ELMO1", "RCSD1", "IQGAP2",
    "C20orf194", "MS4A4A", "LDLRAD4", "FKBP5", "FOXN3",
    
    "FTH1", "SLC2A3", "EPB41L3", "PPARD", "AGFG1", "SRGN", "CEMIP2", "CTSB",
    "DOCK4", "SIPA1L1", "THBS1", "ARHGAP26", "RNF149", "ACSL1", "NAMPT"
]


In [ ]:
sc.pl.dotplot(
    adata, 
    var_names=gene_list,
    groupby='leiden_res0_5',
    dot_max = 0.8,
    standard_scale='var', 
    color_map='RdBu_r' ,
    
   save = 'cluster0vs3genes'
)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
adata.obs

In [ ]:
adata.obs["anatomical_site"] = adata.obs["anatomical_site"].astype(str).str.lower()
adata.obs["disease"] = adata.obs["disease"].astype(str).str.lower()
adata.obs["age_group"] = adata.obs["age_group"].astype(str).str.lower()

In [ ]:
obs_df = adata.obs[['disease', 'leiden_res0_5']]
frequency_df = obs_df.groupby(['disease', 'leiden_res0_5']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(5, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across disease / healthy")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()


In [ ]:
obs_df = adata.obs[['anatomical_site', 'leiden_res0_5']]
frequency_df = obs_df.groupby(['anatomical_site', 'leiden_res0_5']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(5, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across anatomical sites")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
#plt.savefig("samplefreq.pdf", dpi=600)
plt.show()


In [ ]:
obs_df = adata.obs[['age_group', 'leiden_res0_5']]
frequency_df = obs_df.groupby(['age_group', 'leiden_res0_5']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(5, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across age_groups")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
plt.savefig("age.png", dpi=600)
plt.show()


In [ ]:
import os, glob
import scanpy as sc
import anndata as ad
import pandas as pd

input_dir = "/home/yanjun3/atlas/heart_sn"
import scvi
import seaborn as sns
import torch
from rich import print
#from scib_metrics.benchmark import Benchmarker
sc.set_figure_params(figsize=(6, 6),  facecolor="white",
    frameon=True,   dpi = 300, format = 'png')
sc.settings.figdir = "/home/yanjun3/atlas/heart_sn/figures_v3"
sns.set_theme()
torch.set_float32_matmul_precision("high")

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [ ]:
import scanpy as sc
adata = sc.read('/home/yanjun3/atlas/heart_sn/sn_final.h5ad')

In [ ]:
adata

In [ ]:
sc.pl.umap(
    adata,
    color = [
    'LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2',
    'C1QA', 'C1QC', 'CD68', 'CD163', 'HLA-DRA',
    'SPP1', 'APOE',
    'FCGR2A', 'FCGR2B',
    'CLSPN', 'STIL',
    'CD3D', 'CD3E', 'STAB1', 'NEAT1', 'AP2A2', 'leiden_res0_5'
],
    ncols = 2,
    size = 30,
    #save = 'manygenes_umap'
)

In [ ]:
adata.obs["new_anno"] = adata.obs["leiden_res0_5"].map({
    "0": "sn_Lyve1hiNrp1hi",
    "1": "sn_Lyve1+Slc8a1+",
    "2": "sn_Lyve1+_cardiacmuscle_reg",
    "3": "sn_Lyve1+_Tcellregulation",
    "4": "sn_Lyve1-CDK6+",
    "5": "sn_Lyve1-Spp1+Apoe+",
    "6": "sn_sc_cyclingLyve1", 
    "7": "sn_Lyve1+Ms4a4a+",
    "8": "sn_dc",
    "9": "sn_cyclingLyve1"
    
})


In [ ]:
sc.pl.dotplot(
    adata, 
    var_names=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2',
              'C1QA', 'C1QC', 'CD68', 'CD163', 'HLA-DRA',  
               'MS4A4A', 'VSIG4', 'LYZ', 'S100A8', 'SLC8A1', 
               'TTN', 'PRKG1', 'MACF1', 'PTPRJ',

               'CXCL8', 'TNFAIP3', 'NAMPT', 'STAT3', 'NLRP3', 
               "CDK6", "MAPK8", 

               'HBA1', 'HBA2', 'HBB', 
               
               'NRP1', 'STAB1', 'AP2A2', 'CARMIL1', 'DAAM1', 'DNM1', 'PEAK1', 
              'SPP1', 'APOE', 
              'FCGR2A', 'FCGR2B', 'HDAC9', 
              'CLSPN', 'STIL',
              'VEGFC'],
    groupby='leiden_res0_5',
    dot_max = 0.6,
    standard_scale='var', 
    color_map='RdBu_r' ,
    
  save = 'markers.png'
)

In [ ]:
adata.write('/home/yanjun3/atlas/heart_sn/sn_final.h5ad')